# 🔀 Mini pipeline ETL + extracción de IOCs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/clase_en_vivo/live_02_etl_iocs.ipynb)

**Presentar entre las diapositivas 63 y 64** (después de "Veamos un ejemplo").

Construimos un pipeline chiquito: **Extract → Transform → Load** y después
extraemos **IOCs** (indicadores de compromiso) de un texto.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/ETL.png" width="500"/>


## Extract + Transform
Logs crudos de distintas fuentes → un **esquema común** (parseo + normalización).

In [ ]:
# EXTRACT: logs crudos, cada fuente con su formato
crudos = [
    {"fuente": "firewall", "src_ip": "185.220.101.47", "accion": "deny"},
    {"fuente": "windows",  "source_address": "10.0.0.5", "action": "block"},
    {"fuente": "firewall", "src_ip": "185.220.101.47", "accion": "deny"},  # duplicado
]

# TRANSFORM: normalizamos a nombres de campo comunes
def normalizar(e):
    return {
        "source_ip": e.get("src_ip") or e.get("source_address"),
        "accion":    e.get("accion") or e.get("action"),
        "fuente":    e["fuente"],
    }

eventos = [normalizar(e) for e in crudos]

# Deduplicación simple (misma ip + misma acción)
vistos, unicos = set(), []
for e in eventos:
    clave = (e["source_ip"], e["accion"])
    if clave not in vistos:
        vistos.add(clave); unicos.append(e)

print(f"{len(crudos)} crudos -> {len(unicos)} eventos normalizados")
for e in unicos: print(e)

## Extracción de IOCs
Cada tipo de IOC tiene una forma predecible → lo buscamos con **regex**.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/03-pipelines-e-iocs/images/IOCs.png" width="460"/>

In [ ]:
import re

texto = """Reporte: el atacante uso la IP 185.220.101.47 y el dominio
evil-login.com. Adjunto con hash a94a8fe5ccb19ba61c4c0873d391e987982fbbd3
y explota CVE-2024-1234. Contacto: ataque@evil.com"""

PATRONES = {
    "ipv4":   r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
    "sha1":   r"\b[0-9a-fA-F]{40}\b",
    "cve":    r"\bCVE-\d{4}-\d{4,7}\b",
    "email":  r"\b[\w.%+-]+@[\w.-]+\.[a-zA-Z]{2,}\b",
    "dominio": r"\b[a-z0-9-]+\.(?:com|net|org|io)\b",
}

iocs = {tipo: re.findall(p, texto) for tipo, p in PATRONES.items()}
for tipo, valores in iocs.items():
    print(f"{tipo:8}: {valores}")

## Enriquecer + Load
Una IP sola dice poco. La **enriquecemos** (mock) y la cargamos al destino.

In [ ]:
# ENRIQUECIMIENTO (simulado — en real: VirusTotal, MISP, geolocalización)
BLOCKLIST = {"185.220.101.47"}

def enriquecer(ip):
    return {"ip": ip, "en_blocklist": ip in BLOCKLIST,
            "riesgo": "ALTO" if ip in BLOCKLIST else "bajo"}

# LOAD: el resultado final, listo para el SIEM / SOAR
for ip in iocs["ipv4"]:
    print(enriquecer(ip))